# Week 3, Day 5: Minutes for a podcast's planning meeting

**Lesson:** [the Colab notebook](https://colab.research.google.com/drive/1KSMxOCprsl1QRpt_Rq0UqCAyMtPqDQYx) linked from [week3/day5.ipynb](../../../week3/day5.ipynb), *Meeting minutes creator*. **About 45 minutes.**

**What you practice here**

- Transcribing audio with an open model: an automatic-speech-recognition pipeline running Whisper, with timestamps on.
- Transcribing audio with OpenAI's transcription API, asking for plain text back.
- Writing the system and user messages that turn a transcript into minutes, with action items and their owners.
- Generating the minutes with an open model, as on Day 4: the chat template with the generation prompt, `generate`, and decoding only the new tokens.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week3/day5", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson builds a product: it turns the recording of a meeting into minutes. A transcription model, open or
OpenAI's, turns the audio into text, and an open model, Llama 3.2 3B in 4 bits, writes the minutes, with the action
items and who owns each one. Build it for the planning meeting of a WWE history podcast, preparing an episode on
WrestleMania III.

The functions that call a model take it as their first parameter: `transcribe_open` takes transformers' `pipeline`
function, `transcribe_openai` the OpenAI client, and `write_minutes` the model, then its tokenizer. They call these as
the lesson does. The checks hand them stand-ins that record each call and answer with a canned transcript, or canned
minutes.

## What each call should give

```python
transcribe_open(pipeline, "/content/drive/MyDrive/llms/podcast_planning.mp3")
# transcribes the recording with openai/whisper-medium.en on "cuda":
# 'Maya: Welcome, everyone. Today we plan our WrestleMania III episode. ...'

with open("/content/drive/MyDrive/llms/podcast_planning.mp3", "rb") as audio_file:
    transcript = transcribe_openai(client, audio_file)
# one request to gpt-4o-mini-transcribe, which sends the same meeting back as a string

messages = minutes_messages(transcript, "the planning meeting of a WWE history podcast, held online on 14 October 2026")
# [{'role': 'system', 'content': '... meeting minutes ... in markdown ...'},
#  {'role': 'user', 'content': '... the planning meeting of a WWE history podcast ... Maya: Welcome, everyone. ...'}]

write_minutes(model, tokenizer, messages)
# '## Summary\nMaya and Tom planned the WrestleMania III episode. ...\n\n## Action items\n- Tom: Check ...'
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `transcribe_open(pipeline, audio_path, model="openai/whisper-medium.en", device="cuda")`

Transcribes an audio file with an open Whisper model, through an automatic-speech-recognition pipeline. Make the
pipeline with `model` and `device`, and with timestamps turned on: Whisper hears 30 seconds at a time, and the
timestamps let it go through a longer recording. Run it on `audio_path`, and return the text of its result. The lesson
also gives the pipeline a 16-bit dtype; leave that out here, as the checks have no torch, and whisper-medium.en fits on
a T4 without it.

<details><summary>Hint</summary>

```python
speech_to_text = pipeline("automatic-speech-recognition", model=model, device=device, return_timestamps=True)
return speech_to_text(audio_path)["text"]
```

</details>

In [ ]:
def transcribe_open(pipeline, audio_path, model="openai/whisper-medium.en", device="cuda"):
    # Write transcribe_open here, as described above.
    raise NotImplementedError("transcribe_open is not written yet")

In [ ]:
check("transcribe_open")

### 2 · `transcribe_openai(client, audio_file, model="gpt-4o-mini-transcribe")`

Transcribes an audio file with OpenAI's transcription model. `audio_file` is an open binary file, as opening the path
in `"rb"` mode gives. Make one transcription request with `model` and the file, asking for the response as plain text,
every argument by name. With that format, the SDK returns the transcript itself, as a string: return it.

<details><summary>Hint</summary>

```python
return client.audio.transcriptions.create(model=model, file=audio_file, response_format="text")
```

</details>

In [ ]:
def transcribe_openai(client, audio_file, model="gpt-4o-mini-transcribe"):
    # Write transcribe_openai here, as described above.
    raise NotImplementedError("transcribe_openai is not written yet")

In [ ]:
check("transcribe_openai")

### 3 · `minutes_messages(transcript, meeting)`

Builds the messages that ask a model for the minutes of a meeting. Return a list of two dicts: a system message, then
a user message. The system message tells the model that it writes meeting minutes, in markdown. The user message says
what the meeting was, holding `meeting` as it is given, such as "the planning meeting of a WWE history podcast, held
online on 14 October 2026", and holds the whole transcript. Between them, ask for a summary, the discussion points,
the takeaways, and the action items with who owns each one. The wording is yours.

<details><summary>Hint</summary>

Like the lesson's `system_message` and `user_prompt`:

```python
[{"role": "system", "content": "You produce minutes of meetings ... in markdown ..."},
 {"role": "user", "content": f"Below is the transcript of {meeting}. Please write minutes ...\n\n{transcript}"}]
```

</details>

In [ ]:
def minutes_messages(transcript, meeting):
    # Write minutes_messages here, as described above.
    raise NotImplementedError("minutes_messages is not written yet")

In [ ]:
check("minutes_messages")

### 4 · `write_minutes(model, tokenizer, messages, max_new_tokens=2000, device="cuda")`

Writes the minutes with an open model and returns them. Do Day 4's steps in one function: make the tokenizer's pad
token its end-of-sequence token; apply its chat template to `messages`, asking for a PyTorch tensor and the
generation prompt, and move the ids to `device`; then generate, with at most `max_new_tokens` new tokens. Return only
the new tokens, decoded without special tokens. The lesson's cell leaves out the generation prompt: put it in, because
without the cue for the assistant's turn, a model may carry on the transcript instead of writing the minutes.

<details><summary>Hint</summary>

```python
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(device)
outputs = model.generate(inputs, max_new_tokens=max_new_tokens)
return tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
```

</details>

In [ ]:
def write_minutes(model, tokenizer, messages, max_new_tokens=2000, device="cuda"):
    # Write write_minutes here, as described above.
    raise NotImplementedError("write_minutes is not written yet")

In [ ]:
check("write_minutes")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 3, Day 5 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. This needs a GPU, so it runs in Google Colab: push this notebook, then [open it in Colab](https://colab.research.google.com/github/starlest/llm_engineering/blob/main/plan/exercises/week3/day5.ipynb), choose a T4 runtime, and run the cells from the top. In Colab the checks are off, so run them here in your codespace. Write the steps in the 2 cells at the end.

Colab works on its own copy of the notebook. To keep the Try code you write there, use **File > Save a copy in GitHub** (to this notebook's path in your fork), then run `git pull` in your codespace before you commit there again. Your parts are already in your fork, which is what the tracker reads.

Whisper and Llama 3.2 3B in 4 bits both fit on Colab's free T4. Add `HF_TOKEN` and `OPENAI_API_KEY` to Colab's
secrets, and accept the Llama 3.2 terms on its Hugging Face page with the account your token belongs to. OpenAI's
transcription costs well under a cent a minute of audio.

1. Record a one- or two-minute mock planning meeting for a WWE history podcast (or use the lesson's
   `denver_extract.mp3`), and put it on your Google Drive in a folder called `llms`. In the first Try cell, keep the
   `!pip install` line, mount the Drive, set `AUDIO` to the recording's path, log in to Hugging Face with your
   `HF_TOKEN` secret, and make `client = OpenAI(...)` with your `OPENAI_API_KEY` secret: in Colab, the keys come from
   its secrets rather than a `.env` file.
2. Transcribe the recording with `transcribe_open(pipeline, AUDIO)`, and with `transcribe_openai(client, ...)` on the
   opened file. Display both, and compare them.
3. Make the messages with `minutes_messages`, from the transcript you prefer.
4. Load Llama 3.2 3B in 4 bits, as on Day 4: its tokenizer, and the model with the lesson's `BitsAndBytesConfig`.
   Display what `write_minutes(model, tokenizer, messages)` returns, as Markdown.

<details><summary>Stuck? One way to write it</summary>

```python
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6
from google.colab import drive, userdata
from huggingface_hub import login
from IPython.display import Markdown, display
from openai import OpenAI
from transformers import pipeline

drive.mount("/content/drive")
AUDIO = "/content/drive/MyDrive/llms/podcast_planning.mp3"
login(userdata.get("HF_TOKEN"))
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

open_text = transcribe_open(pipeline, AUDIO)
with open(AUDIO, "rb") as audio_file:
    api_text = transcribe_openai(client, audio_file)
display(Markdown(open_text))
display(Markdown(api_text))
```

```python
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

messages = minutes_messages(api_text, "the planning meeting of a WWE history podcast, held online")
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                  bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
display(Markdown(write_minutes(model, tokenizer, messages)))
```

</details>

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6
# Steps 1 and 2: mount the Drive, log in, make the client, and transcribe the recording twice

In [ ]:
# Steps 3 and 4: the messages, then the minutes from Llama 3.2 3B in 4 bits